# Redoubt 2009 mini-observatory workflow (ObsPy)

Notebook series for **2009-03-20 to 2009-03-23 (UTC)**:

1. StationXML subset within **0.2°** of Redoubt (60°29′06″N, 152°44′31″W)  
2. Events from **USGS FDSN Event service (AK catalog)** in same region; plots  
3. Continuous waveforms for those days; **resample-per-ID-per-day then merge**; write daily miniSEED  
4. Helicorder GUI: choose station + day, step forward/back; `dayplot` per Z channel  
5. STA/LTA detection → association → event windows → ObsPy built-in pickers (baseline) → QuakeML

Caching: downloads are skipped if outputs already exist under `ROOT`.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from obspy import UTCDateTime, Stream
from obspy.clients.fdsn import Client

# --- Project config ---
ROOT = os.path.abspath("redoubt_20090320_20090323")
os.makedirs(ROOT, exist_ok=True)

# Redoubt centre: 60°29′06″N 152°44′31″W
REDOUBT_LAT = 60 + 29/60 + 6/3600
REDOUBT_LON = -(152 + 44/60 + 31/3600)

RADIUS_DEG = 0.2

# Time span: 3 full days (end is exclusive)
t0 = UTCDateTime("2009-03-20T00:00:00")
t1 = UTCDateTime("2009-03-23T00:00:00")

# Clients
EARTHSCOPE = Client("EARTHSCOPE")  # waveforms + station
USGS = Client("USGS")              # events (AK)

print("ROOT:", ROOT)
print("Time span:", t0, "to", t1)


In [ ]:
import os
from collections import Counter
from obspy import UTCDateTime

def ensure_dir(path: str) -> str:
    os.makedirs(path, exist_ok=True)
    return path

def file_exists(path: str) -> bool:
    return os.path.exists(path) and os.path.getsize(path) > 0

def list_daily_dates(t0: UTCDateTime, t1: UTCDateTime):
    d = t0
    while d < t1:
        yield d
        d += 24 * 3600

def summarize_duplicates(st):
    ids = [tr.id for tr in st]
    dup = [k for k, v in Counter(ids).items() if v > 1]
    return dup
